# Configuration

In [1]:
%run _dev_setup.py

🔁 Autoreload is ON (IPython detected).
✅ Using recsys from: c:\Users\xusik\Desktop\research\recsys\src\recsys


In [12]:
import os 
from pathlib import Path
from typing import Any, Callable

In [3]:
import polars as pl 
import numpy as np 

import torch 
from torch import nn 
from torch.utils.data import DataLoader, Dataset, random_split

In [4]:
from recsys.data.movielens import MovieLens
from recsys.data.utils import convert_ratings_table_to_matrix, get_ratings_stats
from recsys.utils import IDMapper

# Load data

In [16]:
ml = MovieLens(Path(os.path.join('..', 'datasets')))

data: dict = ml.load('20m')

# Data Preparation

In [ ]:
user_ids: list[str] = data['ratings']['user_id'].unique().to_list()
item_ids: list[str] = data['ratings']['item_id'].unique().to_list()

fields_info: dict[str, dict[str, Any]] = {
    'user_id': {
        'dtype': 'categorical',
        'vocab': user_ids, 
    },
    'item_id': {
        'dtype': 'categorical',
        'vocab': item_ids,
    },
}

n_fields: int = len(fields_info)
cat_dim: int = sum(len(field_info['vocab']) for field_info in fields_info.values() if field_info['dtype'] == 'categorical')

print(f"Number of fields: {n_fields}")
print(f"Total field dimensions: {cat_dim}")

Number of fields: 2
Total field dimensions: 80555


In [ ]:
# Add encoders for categorical features
count = 0
for field, field_dict in fields_info.items():
    if field_dict['dtype'] == 'categorical':
        item_to_idx = IDMapper({item: count + i for i, item in enumerate(field_dict['vocab'])})
        idx_to_item = IDMapper({v: k for k, v in item_to_idx.items()})
        fields_info[field]['encoder'] = item_to_idx
        fields_info[field]['decoder'] = idx_to_item
        count += len(field_dict['vocab'])


In [ ]:
# Label encoder 
def ratings_to_clicks(rating: float, threshold: float = 3.0) -> int: 
    """Convert ratings to binary clicks"""
    return 1 if rating >= threshold else 0

In [ ]:
class MLDataset(Dataset):
    def __init__(
        self, 
        ratings: pl.DataFrame, 
        fields_dict: dict[str, dict[str, Any]], 
        labeler: Callable[[float], int],
        threshold: float = 3.5,
    ): 
        self.ratings = ratings 
        fields: list[str] = [field for field in fields_dict if fields_dict[field]['dtype'] == 'categorical']

        # apply encoders to categorical features
        self.ratings = self.ratings.with_columns(**{
            field: pl.col(field).replace(fields_dict[field]['encoder']).cast(pl.Int32)
            for field in fields_dict if fields_dict[field]['dtype'] == 'categorical'
        })
        
        # convert string values to ints and floats 
        # Use vectorized Polars expression instead of map_elements for much better performance
        # This is ~10-100x faster than map_elements for large datasets
        self.ratings = self.ratings.with_columns(
            (pl.col('rating').cast(pl.Float32) >= threshold).cast(pl.Int32).alias('rating')
        )

        # get dataset 
        self.X: torch.Tensor = torch.tensor(self.ratings[fields].to_numpy(), dtype=torch.int32)
        self.y: torch.Tensor = torch.tensor(self.ratings['rating'].to_numpy(), dtype=torch.float32).unsqueeze(-1)

    def __len__(self) -> int:
        return len(self.ratings)
    
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]

In [ ]:
# hyperparameters 
train_size: float = 0.8
batch_size: int = 10192
val_size: float = 1 - train_size

In [ ]:
dataset = MLDataset(data['ratings'], fields_info, ratings_to_clicks)

# Convert proportions to actual lengths
total_size = len(dataset)
train_len = int(train_size * total_size)
val_len = total_size - train_len

train_set, val_set = random_split(dataset, [train_len, val_len])

In [ ]:
train_dataloader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

# Model

In [ ]:
x_sample, y_sample = next(iter(train_dataloader))

In [ ]:
# Hyperparameters
embed_dim: int = 8

In [ ]:
class DeepFM(nn.Module): 
    def __init__(self, cat_dim: int, embed_dim: int, mlp_dims: list[int]): 
        super().__init__()
        # Embedding layer
        self.cat_embed = nn.Embedding(cat_dim, embed_dim)
        # Factorization machine layer 
        self.o1_fc = nn.Embedding(cat_dim, 1)
        # DNN layer
        self.dnn_mlp = nn.Sequential(
            nn.Linear(n_fields * embed_dim, mlp_dims[0]), 
            *[nn.Sequential(
                nn.BatchNorm1d(dim), 
                nn.ReLU(), 
                nn.Linear(dim, mlp_dims[i + 1])
            ) for i, dim in enumerate(mlp_dims[:-1])], 
            nn.Linear(mlp_dims[-1], 1), 
        )
    
    def forward(self, x: torch.Tensor, y: torch.Tensor = None) -> torch.Tensor: 
        # x has shape (B, n_fields)
        # 1. embedding 
        cat_embeddings = self.cat_embed(x)  # shape: (B, n_fields, embed_dim)
        # 2. factorization machine 
        o1_output = torch.sum(self.o1_fc(x), dim=1)  # shape: (B, 1)
        square_of_sums = torch.sum(cat_embeddings, dim=1) ** 2  # (B, embed_dim)
        sum_of_squares = torch.sum(cat_embeddings ** 2, dim=1)  # (B, embed_dim)
        o2_output = 0.5 * torch.sum(square_of_sums - sum_of_squares, dim=1, keepdim=True)  # (B, 1)
        # 3. DNN 
        dnn_output = self.dnn_mlp(cat_embeddings.view(x.shape[0], -1))  # (B, 1)
        # 4. combine all 
        logits = o1_output + o2_output + dnn_output  # (B, 1)

        if y is not None: 
            loss = nn.functional.binary_cross_entropy_with_logits(logits, y)
            return logits, loss
        else: 
            return logits

# Train

In [ ]:
# hyperparameters 
n_epochs: int = 5
lr: float = 1e-3 
eval_interval: int = 1
device: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
@torch.no_grad()
def estimate_loss(model: nn.Module, dataloaders: dict[str, DataLoader]) -> dict: 
    output: dict = {}
    model.eval()

    for dl_key, dl in dataloaders.items(): 
        losses = torch.zeros(len(dl))
        for i, (x_batch, y_batch) in enumerate(dl): 
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            _, loss = model(x_batch, y_batch)
            losses[i] = loss.item()
        output[dl_key] = losses.mean()

    model.train()
    return output

In [ ]:
model = DeepFM(cat_dim, embed_dim, [64, 32]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for epoch in range(n_epochs): 
    for x_batch, y_batch in train_dataloader: 
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        logits, loss = model(x_batch, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    if epoch % eval_interval == 0: 
        losses = estimate_loss(model, {'train': train_dataloader, 'val': val_dataloader})
        print(f"Epoch {epoch + 1} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")


Epoch 1 | Train Loss: 0.7282 | Val Loss: 0.7311
Epoch 2 | Train Loss: 0.6444 | Val Loss: 0.6498
Epoch 3 | Train Loss: 0.6014 | Val Loss: 0.6097
Epoch 4 | Train Loss: 0.5714 | Val Loss: 0.5824
Epoch 5 | Train Loss: 0.5513 | Val Loss: 0.5646
